# 🎵 ANIMA — Generate & Listen
## GPT-2 53-TET Generation → MPE MIDI → Audio Playback

This notebook:
1. Loads the trained GPT-2 dual-channel model (tokens + EigenSpace)
2. Generates new 53-TET chord sequences with live EigenSpace recomputation
3. Exports to **MPE MIDI** (with proper pitch bends for microtonal playback)
4. Visualizes the piano roll
5. Renders audio inline so you can listen directly in the notebook

In [ ]:
import os
import sys
import json
import time
import importlib
from pathlib import Path
from IPython.display import Audio, display, HTML

import numpy as np
import torch

# Path setup
SRC_DIR = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
ROOT_DIR = SRC_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Source dir: {SRC_DIR}")
print(f"Project root: {ROOT_DIR}")

## 1. Load Model & Vocabulary

In [ ]:
# Import generation module
import importlib
try:
    importlib.reload(sys.modules['10_generate'])
except KeyError:
    pass

# Import via importlib since the filename starts with a number
from importlib import util as _imp_util
spec = _imp_util.spec_from_file_location('gen', str(SRC_DIR / '10_generate.py'))
gen = _imp_util.module_from_spec(spec)
spec.loader.exec_module(gen)

# Load vocabulary
VOCAB_PATH = ROOT_DIR / 'dataset' / 'tokenized' / 'vocab.json'
vocab = gen.Vocabulary(str(VOCAB_PATH))
print(f"Vocabulary: {len(vocab)} tokens")

# Load model from best checkpoint
CHECKPOINT_PATH = ROOT_DIR / 'checkpoints' / 'best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, ckpt = gen.load_checkpoint(str(CHECKPOINT_PATH), device, vocab.vocab_size)

# Load EigenSpace computer
from eigenspace import EigenSpaceComputer
eigen_computer = EigenSpaceComputer(normalize_diss=True)
print(f"\nDevice: {device}")
print("EigenSpace ready")
print("Model loaded ✓")

## 2. Generation Settings
Adjust these parameters to control the generation.

In [ ]:
# ── Generation parameters ──
MAX_TOKENS    = 512       # Number of tokens to generate
TEMPERATURE   = 0.85      # Lower = more focused, Higher = more creative
TOP_K         = 50        # Top-k filtering (None to disable)
TOP_P         = 0.95      # Nucleus sampling threshold (None to disable)
SEED          = None      # Set to an integer for reproducible results

# ⚠️ IMPORTANT: Use 'chord' mode (recomputes eigenspace after every token)
# This matches the training behavior where each token had eigenspace of the
# partial chord built so far. Using defaults causes single-note generation!
EIGEN_MODE    = 'chord'   # 'chord' = recompute after each token (REQUIRED!)
                          # 'bar'   = recompute per bar (will generate single notes)
                          # 'none'  = static defaults (will generate single notes)

# ── Prompt (set to None for unconditional generation) ──
# You can provide starting tokens to guide the model:
PROMPT = None  # Unconditional — let the model compose freely

# Examples of prompts you can try:
# PROMPT = '<start> BAR'                           # Start from a bar line
# PROMPT = '<start> CHORD_START DUR_4.0'           # Start with a whole-note chord
# PROMPT = '<start> CHORD_START DUR_2.0'            # Start with a 2-beat chord

# ── Audio rendering ──
PLAYBACK_SPEED = 1.2     # Speed multiplier for audio playback  
WAVEFORM       = 'sine'  # 'sine', 'triangle', 'square', or 'clarinet'
REVERB         = 20      # Reverb amount (0-100)
MIDI_TEMPO     = 120     # BPM for MIDI export

# ── Output ──
OUTPUT_DIR = ROOT_DIR / 'dataset' / 'generated'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Settings configured ✓")
print(f"  Max tokens:  {MAX_TOKENS}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Top-k:       {TOP_K}")
print(f"  Top-p:       {TOP_P}")
print(f"  Eigen mode:  {EIGEN_MODE}")
print(f"  Waveform:    {WAVEFORM}")
print(f"  Reverb:      {REVERB}%")
print(f"  Output dir:  {OUTPUT_DIR}")

## 3. Generate!

In [ ]:
# Set seed if specified
if SEED is not None:
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    print(f"Seed: {SEED}")

# Build prompt
if PROMPT is not None:
    prompt_tokens = PROMPT.strip().split()
    unknown = [t for t in prompt_tokens if t not in vocab.token_to_id]
    if unknown:
        print(f"⚠️ Unknown tokens removed: {unknown}")
        prompt_tokens = [t for t in prompt_tokens if t in vocab.token_to_id]
    prompt_ids = vocab.encode(prompt_tokens)
else:
    prompt_tokens = ['<start>']
    prompt_ids = [vocab.start_id]

print(f"Prompt: {' '.join(prompt_tokens)}")
print(f"Generating {MAX_TOKENS} tokens...")

t0 = time.time()
gen_ids, gen_tokens = gen.generate_with_eigenspace(
    model, vocab, eigen_computer,
    prompt_ids=prompt_ids,
    max_new_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    device=device,
    recompute_interval=EIGEN_MODE,
)
dt = time.time() - t0
new_count = len(gen_ids) - len(prompt_ids)

print(f"\n✓ Generated {new_count} tokens in {dt:.2f}s ({new_count/dt:.0f} tok/s)")

# Summary
chords = gen.extract_chords_summary(gen_tokens)
n_bars = sum(1 for t in gen_tokens if t == 'BAR')
print(f"  Chords: {len(chords)}")
print(f"  Bars:   {n_bars}")

## 4. View Generated Sequence

In [ ]:
# Verify chord structure
from collections import Counter
pitch_counts = Counter(len(c['pitches']) for c in chords)

print("─── Chord Analysis ───")
print(f"Total chords: {len(chords)}")
print(f"\nNotes per chord:\n")
for n_pitches in sorted(pitch_counts.keys()):
    pct = 100 * pitch_counts[n_pitches] / len(chords)
    bar = '█' * int(pct / 2)
    print(f"  {n_pitches} notes: {pitch_counts[n_pitches]:3d} ({pct:5.1f}%) {bar}")

avg = sum(k * v for k, v in pitch_counts.items()) / len(chords)
print(f"\nAverage: {avg:.2f} notes/chord (training: 5.08) ✓")

# Show first few chords
print("\n─── First 5 Chords ───")
for i, chord in enumerate(chords[:5], 1):
    print(f"  Chord {i}: {len(chord['pitches'])} notes, dur={chord['duration']}")

print("\n─── Readable Format Sample ───")
print(gen.format_as_readable(gen_tokens[:150]) + "...")

## 5. Export to MPE MIDI
Convert the 53-TET pitches to MPE MIDI (one channel per note, with pitch bends for microtuning).
This is the same format used in the training dataset, so `play_mpe` can render it correctly.

In [ ]:
import mido

def export_53tet_mpe_midi(tokens, output_path, tempo=120, velocity=80, pitch_offset=106):
    """
    Export generated tokens to MPE MIDI with proper 53-TET pitch bends.
    
    Each note gets its own MIDI channel with a pitch bend that shifts
    the 12-TET base note to the exact 53-TET frequency. This matches
    the format used in the training data (generate_53tet_dataset.py)
    and is correctly rendered by play_mpe.py.
    
    53-TET → 12-TET mapping:
      midi_note = round(step_53 * 12 / 53)
      deviation_cents = (step_53 * 12 / 53 - midi_note) * 100
      pitch_bend = deviation_cents / 200 * 8192  (±2 semitone range)
    """
    mid = mido.MidiFile(type=0, ticks_per_beat=480)
    track = mido.MidiTrack()
    mid.tracks.append(track)
    
    # Tempo
    us_per_beat = int(60_000_000 / tempo)
    track.append(mido.MetaMessage('set_tempo', tempo=us_per_beat))
    
    tpb = mid.ticks_per_beat
    
    # Extract chords from tokens
    chords = gen.extract_chords_summary(tokens)
    
    if not chords:
        print("⚠️ No chords found in the generated sequence.")
        return None
    
    # MPE: each note gets its own channel (channels 1-15, 0 reserved for global)
    next_channel = 1
    max_channels = 15
    
    for chord in chords:
        if not chord['pitches'] or chord['duration'] is None:
            continue
        
        dur_ticks = int(chord['duration'] * tpb)
        notes_in_chord = []
        
        for p53_abs in chord['pitches']:
            # p53_abs is the absolute 53-TET step (e.g. 265 = C4, 212 = ~Ab3)
            # Convert to nearest 12-TET MIDI note + pitch bend for MPE
            # Matches step53_to_midi_and_bend() in 05_midi_mpe_tokenization.py
            midi_note = round(p53_abs * 12.0 / 53.0)
            midi_note = max(0, min(127, midi_note))
            
            # Residual deviation in cents -> pitch bend (+/-200 cents range)
            residual_steps = p53_abs - midi_note * 53.0 / 12.0
            deviation_cents = residual_steps * 1200.0 / 53.0
            bend_value = int(round(deviation_cents / 200.0 * 8192))
            bend_value = max(-8192, min(8191, bend_value))
            
            # Assign MPE channel
            ch = next_channel
            next_channel = (next_channel % max_channels) + 1
            
            notes_in_chord.append((midi_note, bend_value, ch))
        
        # Write pitch bends and note-ons (time=0 for simultaneous)
        for i, (note, bend, ch) in enumerate(notes_in_chord):
            track.append(mido.Message('pitchwheel', pitch=bend, channel=ch, time=0))
            track.append(mido.Message('note_on', note=note, velocity=velocity, channel=ch, time=0))
        
        # Note-offs after duration
        for i, (note, bend, ch) in enumerate(notes_in_chord):
            track.append(mido.Message(
                'note_off', note=note, velocity=0, channel=ch,
                time=dur_ticks if i == 0 else 0  # duration on first, 0 for rest
            ))
            # Reset pitch bend
            track.append(mido.Message('pitchwheel', pitch=0, channel=ch, time=0))
    
    # Save
    os.makedirs(os.path.dirname(str(output_path)) or '.', exist_ok=True)
    mid.save(str(output_path))
    
    n_notes = sum(len(c['pitches']) for c in chords if c['pitches'])
    print(f"✓ Exported MPE MIDI: {output_path}")
    print(f"  {len(chords)} chords, {n_notes} notes, {n_bars} bars")
    return str(output_path)


# Generate timestamp for filename
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
midi_filename = f"generated_{timestamp}.mid"
midi_path = OUTPUT_DIR / midi_filename

# Export
result = export_53tet_mpe_midi(
    gen_tokens, midi_path,
    tempo=MIDI_TEMPO,
    pitch_offset=vocab.config.get('pitch_offset', 106),
)

if result:
    print(f"\nFile: {midi_path.name}")

## 6. Visualize Piano Roll

In [ ]:
import midi_viz as mv
importlib.reload(mv)

if midi_path.exists():
    print(f"Visualizing: {midi_path.name}")
    fig = mv.visualize_midi(str(midi_path), speed=PLAYBACK_SPEED, max_duration=120)
    if fig:
        fig.update_layout(
            title=f"Generated Sequence — {len(chords)} chords, {n_bars} bars",
            height=500,
        )
        fig.show()
else:
    print("⚠️ No MIDI file to visualize")

## 7. Listen 🎧
Render the MPE MIDI to audio and play inline.

In [ ]:
import play_mpe as pm
importlib.reload(pm)

if midi_path.exists():
    print(f"Rendering: {midi_path.name}")
    print(f"  Waveform: {WAVEFORM} | Reverb: {REVERB}% | Speed: {PLAYBACK_SPEED}x")
    
    audio_data, sr = pm.render_mpe_to_audio_data(
        str(midi_path),
        speed=PLAYBACK_SPEED,
        waveform=WAVEFORM,
        reverb=REVERB,
    )
    
    if audio_data is not None:
        display(Audio(audio_data, rate=sr))
    else:
        print("⚠️ No audio rendered — check the MIDI file")
else:
    print("⚠️ No MIDI file found")

## 8. Batch Generate
Generate multiple samples and listen to each one.

In [ ]:
NUM_SAMPLES = 3

for i in range(NUM_SAMPLES):
    print(f"\n{'='*60}")
    print(f"Sample {i+1}/{NUM_SAMPLES}")
    print(f"{'='*60}")
    
    # Generate
    ids, tokens = gen.generate_with_eigenspace(
        model, vocab, eigen_computer,
        prompt_ids=[vocab.start_id],
        max_new_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        top_k=TOP_K, top_p=TOP_P,
        device=device,
        recompute_interval=EIGEN_MODE,
    )
    
    sample_chords = gen.extract_chords_summary(tokens)
    sample_bars = sum(1 for t in tokens if t == 'BAR')
    print(f"  {len(sample_chords)} chords, {sample_bars} bars, {len(ids)} tokens")
    
    # Readable view
    print(gen.format_as_readable(tokens))
    
    # Export to MPE MIDI
    sample_path = OUTPUT_DIR / f"generated_{timestamp}_sample{i+1}.mid"
    export_53tet_mpe_midi(
        tokens, sample_path,
        tempo=MIDI_TEMPO,
        pitch_offset=vocab.config.get('pitch_offset', 106),
    )
    
    # Render and play
    if sample_path.exists():
        audio_data, sr = pm.render_mpe_to_audio_data(
            str(sample_path), speed=PLAYBACK_SPEED,
            waveform=WAVEFORM, reverb=REVERB,
        )
        if audio_data is not None:
            display(Audio(audio_data, rate=sr))

print(f"\n{'='*60}")
print(f"All {NUM_SAMPLES} samples generated in {OUTPUT_DIR}")

## 9. Compare with Training Data
Listen to a random file from the original dataset for comparison.

In [ ]:
import random

DATASET_MIDI_PATH = ROOT_DIR / 'dataset' / 'midi_files' / '53_tet_mpe'

if DATASET_MIDI_PATH.exists():
    midi_files = sorted(DATASET_MIDI_PATH.glob('*.mid'))
    if midi_files:
        ref_file = random.choice(midi_files)
        print(f"Reference file: {ref_file.name}")
        
        # Visualize
        fig = mv.visualize_midi(str(ref_file), speed=PLAYBACK_SPEED, max_duration=60)
        if fig:
            fig.update_layout(title=f"Training Data — {ref_file.stem}", height=400)
            fig.show()
        
        # Play
        audio_data, sr = pm.render_mpe_to_audio_data(
            str(ref_file), speed=PLAYBACK_SPEED,
            waveform=WAVEFORM, reverb=REVERB,
        )
        if audio_data is not None:
            display(Audio(audio_data, rate=sr))
    else:
        print("No MIDI files found in dataset")
else:
    print(f"Dataset MIDI path not found: {DATASET_MIDI_PATH}")
    print("(This is fine — you can still generate and listen above)")